# APIM ❤️ Passthrough

![flow](https://docs.azure.cn/en-us/api-management/media/authentication-authorization-overview/oauth-token-gateway.png)

Playground to try the [OAuth 2.0 authorization feature](https://learn.microsoft.com/azure/api-management/api-management-authenticate-authorize-azure-openai#oauth-20-authorization-using-identity-provider) using identity provider to enable more fine-grained access to AI Foundry models by particular users or client.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`... 


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the models and versions according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [20]:
import os, sys, json
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"demo-{deployment_name}" # change the name to match your naming style
resource_group_location = "swedencentral"  # change to your preferred Azure region

apim_sku = 'Standardv2'  # options: Developer, Basic, Standard, Premium

apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

app_registration_name = f"{deployment_name}-new-app" # name of the app that will be registered in Microsoft Entra ID

passthrough_backend_url = "https://webhook.site/c04c5082-8c78-4820-84ff-deeb961c4c2d"
passthrough_api_type = "ServiceNow"  # options: ServiceNow, Cognigy, etc.

auth_header_name = "X-Api-Key"
auth_key = "SuperSecretApiKey123"

# Custom domain configuration
custom_domain_name = "apimdemo.pankaagr.cloud"  # change to your custom domain

utils.print_ok('Notebook initialized')

✅ Notebook initialized ⌚ 12:13:18.713426 


<a id='1'></a>
### 1️⃣ Create the App Registration in Microsoft Entra ID
The following command creates a client application registration

In [21]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

client_id = None
output = utils.run(f"az ad app list --filter \"displayName eq '{app_registration_name}'\"", f"Retrieved app registration with name {app_registration_name}", "Failed to get the app registration")
if output.success and output.json_data:
    client_id = output.json_data[0]['appId']
else:
    output = utils.run(f"az ad app create --display-name {app_registration_name} --is-fallback-public-client true", f"Created app registration with name {app_registration_name}", "Failed to create the app registration")
    if output.success and output.json_data:
        client_id = output.json_data['appId']

print(f"👉🏻 Client Id: {client_id}")


⚙️ Running: az account show 
✅ Retrieved az account ⌚ 12:13:26.149318 :0s]
👉🏽 Current user: admin@MngEnvMCAP035014.onmicrosoft.com
👉🏽 Tenant ID: 0134b439-98e4-4dd1-9a16-d00d553b93e4
👉🏽 Subscription ID: 423d99c5-cef2-4f93-980a-2811e651fd11
⚙️ Running: az ad app list --filter "displayName eq 'passthrough-access-control-new-app'" 
✅ Retrieved app registration with name passthrough-access-control-new-app ⌚ 12:13:27.710108 :1s]
⚙️ Running: az ad app create --display-name passthrough-access-control-new-app --is-fallback-public-client true 
✅ Created app registration with name passthrough-access-control-new-app ⌚ 12:13:30.251972 :2s]
👉🏻 Client Id: f844b415-e53b-46bc-af23-f428c2b1b2c7


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.


In [41]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "passthroughApi": { "value": passthrough_backend_url },
        "passthroughApiType": { "value": passthrough_api_type },
        "authHeaderName": { "value":  auth_header_name },
        "authKey": { "value": auth_key },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "tenantId": { "value": tenant_id },
        "clientId": { "value": client_id }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

⚙️ Running: az group show --name demo-passthrough-access-control 
👉🏽 Using existing resource group 'demo-passthrough-access-control'
⚙️ Running: az deployment group create --name passthrough-access-control --resource-group demo-passthrough-access-control --template-file main.bicep --parameters params.json 
✅ Deployment 'passthrough-access-control' succeeded ⌚ 14:49:25.263086 :12s]


<a id='create-cert'></a>
### 🔐 Create Self-Signed Certificate

Generate a self-signed SSL certificate for the custom domain to be used with APIM.

In [32]:
import hashlib
import os

notebook_dir = os.path.dirname(globals()['__vsc_ipynb_file__'])
os.chdir(notebook_dir)  # Change to notebook directory

# Generate certificate file names based on domain
cert_base_name = custom_domain_name.replace('.', '-')
cert_file = f"{cert_base_name}.pem"
key_file = f"{cert_base_name}-key.pem"
pfx_file = f"{cert_base_name}.pfx"

# Check if certificate already exists
if os.path.exists(pfx_file):
    utils.print_info(f"Certificate '{pfx_file}' already exists")
else:
    utils.print_info(f"Generating self-signed certificate for {custom_domain_name}...")
    
    # Generate private key and certificate
    output = utils.run(
        f'openssl req -x509 -newkey rsa:4096 -keyout {key_file} -out {cert_file} -days 365 -nodes -subj "/CN={custom_domain_name}" -addext "subjectAltName=DNS:{custom_domain_name},DNS:*.{custom_domain_name}"',
        f"Generated certificate for {custom_domain_name}",
        f"Failed to generate certificate"
    )
    
    if output.success:
        # Convert to PFX format
        output = utils.run(
            f'openssl pkcs12 -export -out {pfx_file} -inkey {key_file} -in {cert_file} -passout pass:',
            f"Converted certificate to PFX format: {pfx_file}",
            f"Failed to convert certificate to PFX"
        )
        
        if output.success:
            utils.print_ok(f"✅ Certificate files created: {cert_file}, {key_file}, {pfx_file}")

# Generate a unique Key Vault name based on subscription ID
subscription_hash = hashlib.md5(subscription_id.encode()).hexdigest()[:6]
keyvault_name = f"kv-apim-{subscription_hash}"

# Create Key Vault if it doesn't exist
output = utils.run(
    f"az keyvault show --name {keyvault_name} --resource-group {resource_group_name}",
    f"Key Vault '{keyvault_name}' already exists",
    f"Key Vault '{keyvault_name}' not found, will create it"
)

if not output.success:
    # Create the Key Vault
    output = utils.run(
        f"az keyvault create --name {keyvault_name} --resource-group {resource_group_name} --location {resource_group_location} --enable-rbac-authorization false",
        f"Created Key Vault '{keyvault_name}'",
        f"Failed to create Key Vault '{keyvault_name}'"
    )

if output.success:
    utils.print_info(f"Key Vault Name: {keyvault_name}")
    
    # Upload the certificate to Key Vault
    cert_name = cert_base_name
    
    output = utils.run(
        f"az keyvault certificate import --vault-name {keyvault_name} --name {cert_name} --file {pfx_file}",
        f"Certificate '{cert_name}' uploaded successfully to Key Vault",
        f"Failed to upload certificate to Key Vault"
    )
    
    if output.success and output.json_data:
        cert_id = output.json_data.get('id')
        cert_thumbprint = output.json_data.get('x509Thumbprint')
        utils.print_info(f"Certificate ID: {cert_id}")
        utils.print_info(f"Certificate Thumbprint: {cert_thumbprint}")
        utils.print_ok(f"✅ Certificate ready for APIM custom domain configuration")

👉🏽 Generating self-signed certificate for apimdemo.pankaagr.cloud...
⚙️ Running: openssl req -x509 -newkey rsa:4096 -keyout apimdemo-pankaagr-cloud-key.pem -out apimdemo-pankaagr-cloud.pem -days 365 -nodes -subj "/CN=apimdemo.pankaagr.cloud" -addext "subjectAltName=DNS:apimdemo.pankaagr.cloud,DNS:*.apimdemo.pankaagr.cloud" 
✅ Generated certificate for apimdemo.pankaagr.cloud ⌚ 12:31:23.497247 :1s]
⚙️ Running: openssl pkcs12 -export -out apimdemo-pankaagr-cloud.pfx -inkey apimdemo-pankaagr-cloud-key.pem -in apimdemo-pankaagr-cloud.pem -passout pass: 
✅ Converted certificate to PFX format: apimdemo-pankaagr-cloud.pfx ⌚ 12:31:23.681447 :0s]
✅ ✅ Certificate files created: apimdemo-pankaagr-cloud.pem, apimdemo-pankaagr-cloud-key.pem, apimdemo-pankaagr-cloud.pfx ⌚ 12:31:23.681707 
⚙️ Running: az keyvault show --name kv-apim-f7a8ee --resource-group demo-passthrough-access-control 
✅ Key Vault 'kv-apim-f7a8ee' already exists ⌚ 12:31:25.125000 :1s]
👉🏽 Key Vault Name: kv-apim-f7a8ee
⚙️ Running: 

<a id='4'></a>
### 4️⃣ Get the deployment outputs

We are now at the stage where we only need to retrieve the gateway URL and the subscription before we are ready for testing.

In [4]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key") # default api key to the first subscription key

⚙️ Running: az deployment group show --name passthrough-access-control -g demo-passthrough-access-control 
✅ Retrieved deployment: passthrough-access-control ⌚ 11:45:06.729588 :1s]
👉🏽 Log Analytics Id: f3f9d1cb-d5d0-481c-b053-43ffc784f3a3
👉🏽 APIM Service Id: /subscriptions/423d99c5-cef2-4f93-980a-2811e651fd11/resourceGroups/demo-passthrough-access-control/providers/Microsoft.ApiManagement/service/demo-apim-vuzo7oqc4xzcc
👉🏽 APIM API Gateway URL: https://demo-apim-vuzo7oqc4xzcc.azure-api.net
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****921f


<a id='5'></a>
### 5️⃣ Create a device flow to get the access token

Notes for fine grained authorization:
- The APIM [JWT validation policy](https://learn.microsoft.com/azure/api-management/validate-azure-ad-token-policy) can check for specific claims (that needs to exist in the token) and apply fine-grained authorization.
- Group claims is a typical method. You can use this approach to drive authorization. However, when the user is a member of too many groups, the `groups` will be excluded from the token due to limitations in token size.
- An alternative is to configure app role definitions and assign users/groups to app roles. This Zero Trust developer best practice improves flexibility and control while increasing application security with least privilege. [Learn more](https://learn.microsoft.com/security/zero-trust/develop/configure-tokens-group-claims-app-roles).
- To obtain the `roles` claim, navigate to the "Expose an API" section of the App Registration. Add the Application ID URI and a scope. Then, copy the full scope `(app://<id>/scope)` and replace in the scope array below.
- Navigate to the "App Roles" blade and create an App Role (ex: OpenAI.ChatCompletion) for Users/Groups members. Then assign the testing user or group to the App Role.   
- After logging in, use https://jwt.io/ to decode the `access_token` variable and verify that the `roles` are being sent.
- With the above configuration, you can add the following fragment to the APIM policy to verify that the user belongs to a specific App Role:
```
            <required-claims>
                <claim name="roles" match="any">
                    <value>OpenAI.ChatCompletion</value>
                </claim>
            </required-claims>
```



In [42]:
import json
import msal

app = msal.PublicClientApplication(client_id, authority = "https://login.microsoftonline.com/" + tenant_id)
# scope = ["User.Read"]
scope = ["api://f844b415-e53b-46bc-af23-f428c2b1b2c7/ServiceNowScope"]

flow = app.initiate_device_flow(scopes = scope)

if "user_code" not in flow:
    raise ValueError(
        "Fail to create device flow. Err: %s" % json.dumps(flow, indent = 4))

print(flow["message"])

To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code EUG4EXXWR to authenticate.


<a id='6'></a>
### 6️⃣ Acquire the token and query the graph API

In [ ]:
import requests, base64, json
import subprocess
import platform

result = app.acquire_token_by_device_flow(flow)

if "access_token" in result:
    access_token = result['access_token']
    # Calling graph using the access token
    header, payload, signature = access_token.split('.')
    def pad(b): return b + '=' * (-len(b) % 4)
    print("Decoded JWT Header and Payload:")
    print(json.dumps(json.loads(base64.urlsafe_b64decode(pad(header)).decode('utf-8')), indent=4))
    print(json.dumps(json.loads(base64.urlsafe_b64decode(pad(payload)).decode('utf-8')), indent=4))

    graph_data = requests.get(  # Use token to call downstream service
        "https://graph.microsoft.com/v1.0/me",
        headers={'Authorization': 'Bearer ' + access_token},).json()
    print("Graph API call result: %s" % json.dumps(graph_data, indent = 2))
    
    # Copy access token to clipboard (macOS only)
    if platform.system() == 'Darwin':
        try:
            subprocess.run('pbcopy', text=True, input=access_token, check=True)
            print("\n✅ Access token copied to clipboard")
        except Exception as e:
            print(f"\n⚠️ Could not copy to clipboard: {e}")
    
    print("\nAccess Token:\n%s\n" % access_token)
    # print(access_token) # Use a tool like https://jwt.io/ to decode the access token and see its contents
else:
    print(result.get("error"))
    print(result.get("error_description"))
    print(result.get("correlation_id"))  # You may need this when reporting a bug


Decoded JWT Header and Payload:
{
    "typ": "JWT",
    "alg": "RS256",
    "x5t": "rtsFT-b-7LuY7DVYeSNKcIJ7Vnc",
    "kid": "rtsFT-b-7LuY7DVYeSNKcIJ7Vnc"
}
{
    "aud": "api://f844b415-e53b-46bc-af23-f428c2b1b2c7",
    "iss": "https://sts.windows.net/0134b439-98e4-4dd1-9a16-d00d553b93e4/",
    "iat": 1763041570,
    "nbf": 1763041570,
    "exp": 1763045997,
    "acr": "1",
    "aio": "AXQAi/8aAAAAakGzBpt0MpNq1yRSx0WkFv0bUYBxis0dd6ecCQgS6nCQGU5xXnT8+g0wvWRTddCSRCYy+VVNI5PLWccuq13orhMXRtDs3CBjWbJIWX93LO1bLFIDpevC2Uxf/PlMM5CvVnLjogicPVSwc0uBa9eHiQ==",
    "amr": [
        "pwd",
        "mfa"
    ],
    "appid": "f844b415-e53b-46bc-af23-f428c2b1b2c7",
    "appidacr": "0",
    "family_name": "Administrator",
    "given_name": "System",
    "ipaddr": "84.215.107.187",
    "name": "System Administrator",
    "oid": "a701b535-3646-4256-88df-98c92af05b33",
    "rh": "1.AWMBObQ0AeSY0U2aFtANVTuT5BW0RPg75bxGryP0KMKxssdjAZxjAQ.",
    "roles": [
        "ServiceNow"
    ],
    "scp": "ServiceNowSc

<a id='requests'></a>
### 🧪 Test the API using the access token

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to debug the policy.

In [40]:
url = apim_resource_gateway_url + "/passthrough/cognigy/testing-path"

print(f"Making request to URL: {url}")

messages = { "messages": [
    {"role": "system", "content": "You are a sarcastic unhelpful assistant."},
    {"role": "user", "content": "Can you tell me the time, please?"}
]}

response = requests.post(url, headers = {'Authorization': 'Bearer ' + access_token}, json = messages)
utils.print_response_code(response)

if (response.status_code == 200):
    print(response.text)
else:
    print(response.text)


Making request to URL: https://demo-apim-vuzo7oqc4xzcc.azure-api.net/passthrough/cognigy/testing-path
Response status: 200 - OK
{
"status":"ok"
}


<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.